# Actuator dynamics: powertrain step responseWheels raised, so this is the closed-loop response (setpoint -> actual speed withthe PID inside), which is exactly what the NMPC will command in Phase 9. The timeconstant depends on the PID gains: if Kp/Ki change, this identification is stale.Four speed steps: 0.3, 0.6, 1.0, 1.3 m/s. First-order + delay fit on each. Stepsare detected directly from wheel speed (the /drive command was not captured inthe bag, but the response is all we need).

In [ ]:
import sys
sys.path.append('../scripts')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import bagtools as bt

R_WHEEL = 0.03415

dfs = bt.load('../data/actuator_steps_parquet')
dfs = bt.zero_time(dfs)
js = dfs['joint_states']
js['w'] = (js['wl'] + js['wr']) / 2.0

## Detect the step windows from wheel speedA step starts when wheel speed rises from rest. We label each step by the plateauit settles to.

In [ ]:
w = js['w'].values
t = js['t'].values
moving = w > 2.0                      # threshold rad/s
edges = []
for i in range(1, len(moving)):
    if moving[i] and not moving[i-1]:
        seg = w[i:i+150]             # ~3 s at 50 Hz
        plateau = np.median(seg[-30:]) if len(seg) > 30 else w[i]
        edges.append((t[i], plateau * R_WHEEL))

print("detected steps:")
for te, a in edges:
    print(f"  t = {te:6.2f} s   plateau ~ {a:.2f} m/s   ({a/R_WHEEL:.1f} rad/s)")

## Fit each stepFirst-order plus dead time:$$\omega(t) = \omega_\infty \left(1 - e^{-(t - t_d)/\tau}\right), \quad t > t_d$$$\tau$ = time constant, $t_d$ = pure delay, $\omega_\infty$ = steady value.Time is measured from the detected rising edge.

In [ ]:
def step_model(t, w_inf, tau, td):
    y = w_inf * (1.0 - np.exp(-(t - td) / tau))
    return np.clip(y, 0.0, None)

results = []
n = len(edges)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(12, 3.2 * ((n + 1) // 2)))
axes = np.atleast_1d(axes).flat

for k, (t_edge, cmd_amp) in enumerate(edges):
    seg = bt.window(js, t_edge - 0.3, t_edge + 3.0).copy()
    t_rel = (seg['t'] - t_edge).values
    y = seg['w'].values
    m = t_rel > 0

    try:
        p, _ = curve_fit(step_model, t_rel[m], y[m],
                         p0=[y[m].max(), 0.15, 0.05],
                         bounds=([0, 0.01, 0], [np.inf, 2.0, 0.5]))
        w_inf, tau, td = p
    except Exception as e:
        print(f"step {k}: fit failed ({e})")
        continue

    results.append({'cmd_mps': round(cmd_amp, 2), 'w_inf': w_inf,
                    'tau': tau, 'td': td})

    ax = axes[k]
    ax.plot(t_rel[m], y[m], '.', ms=3, label='measured')
    tt = np.linspace(0, t_rel.max(), 200)
    ax.plot(tt, step_model(tt, *p), 'r', lw=2, label='fit')
    ax.set_title(f'{cmd_amp:.2f} m/s: tau={tau*1000:.0f} ms, td={td*1000:.0f} ms')
    ax.set_xlabel('t [s]'); ax.set_ylabel('w [rad/s]')
    ax.grid(alpha=0.3); ax.legend()

plt.tight_layout(); plt.show()

## Summary: is the actuator linear?If tau is roughly constant across amplitudes, one first-order model works. If taugrows with amplitude, the actuator saturates (expected on the top step, which hitsthe motor free-speed limit with no torque reserve).

In [ ]:
t = pd.DataFrame(results)
t['tau_ms'] = (t['tau'] * 1000).round(0)
t['td_ms'] = (t['td'] * 1000).round(0)
print(t[['cmd_mps', 'w_inf', 'tau_ms', 'td_ms']].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t['cmd_mps'], t['tau_ms'], 'o-')
ax.set_xlabel('commanded speed [m/s]'); ax.set_ylabel('tau [ms]')
ax.set_title('time constant vs amplitude'); ax.grid(alpha=0.3)
plt.show()

## For vehicle_params.yamlTake tau and td from the low/mid amplitude steps (before saturation) as thenominal closed-loop actuator model. Note in the yaml that these are closed-loopvalues tied to the current PID gains.The servo response is not identified here: /steering_angle publishes thecommanded angle, not the physical one. If servo dynamics matter in Phase 9,refine with high-speed video.